# Post-Phase14 / H3 Proposal-Supply Research Pipeline

Master index for the stepwise research record. This notebook links the completed
post-Phase14 causal/design steps and clearly marks pending future H3 steps. It does not
duplicate large result tables, train models, run inference, evaluate H3, access TEST, or
start Phase 15.

In [1]:
from pathlib import Path
import hashlib
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def discover_repo(start=None):
    candidates = []
    if start is not None:
        candidates.append(Path(start).resolve())
    candidates.extend([Path.cwd().resolve(), Path.cwd().resolve().parent])
    try:
        candidates.append(Path(__file__).resolve())
    except NameError:
        pass
    for candidate in candidates:
        current = candidate if candidate.is_dir() else candidate.parent
        for root in (current, *current.parents):
            if all((root / name).is_dir() for name in ("scripts", "reports", "manifests", "notebooks")):
                return root
    fallback = Path(r"D:/Softfors/jup")
    if all((fallback / name).is_dir() for name in ("scripts", "reports", "manifests", "notebooks")):
        return fallback
    raise RuntimeError("Repository root not found: expected scripts/, reports/, manifests/, notebooks/.")

ROOT = discover_repo()
AUDITS = ROOT / "reports" / "audits"
TABLES = ROOT / "reports" / "tables"

def load_json(relative):
    with (ROOT / relative).open("r", encoding="utf-8") as handle:
        return json.load(handle)

def read_csv(relative):
    return pd.read_csv(ROOT / relative)

def sha(relative):
    return hashlib.sha256((ROOT / relative).read_bytes()).hexdigest()

print("Repository root:", ROOT)
print("No training or inference is executed by this notebook.")

Repository root: D:\Softfors\jup
No training or inference is executed by this notebook.


In [2]:
import hashlib
entries = [
    ("01", "01_proposal_root_cause_diagnostic.ipynb", "proposal/localizer root-cause diagnostic", "COMPLETE", "F_MIXED_BOTTLENECK"),
    ("02", "02_h3_axis_priority_diagnostic.ipynb", "A-only vs C-only causal priority", "COMPLETE", "H3_DESIGN_ELIGIBLE__AXIS_PROPOSAL_SUPPLY_RECALL"),
    ("03", "03_h3_design_and_pretraining_binding.ipynb", "H3 design, corrections, calibration, binding", "COMPLETE", "H3_FINAL_PRETRAINING_BINDING_COMPLETE__AWAITING_TRAINING_AUTHORIZATION"),
    ("04", "04_h3_training.ipynb", "H3 training record", "COMPLETE", "H3_TRAINING_COMPLETE__CHECKPOINT_SELECTION_REPLAY_PASS__SUPPLY_EVALUATION_PENDING"),
    ("05", "05_h3_supply_axis_evaluation.ipynb", "H3 supply-axis evaluation (metric-domain RECONCILED)", "COMPLETE / RECONCILED", "H3_SUPPLY_METRIC_DOMAIN_RECONCILED__DOWNSTREAM_R_H3_PENDING"),
    ("06", "06_h3_downstream_evaluation.ipynb", "H3 frozen downstream evaluation", "COMPLETE", "H3_DOWNSTREAM_EVALUATION_COMPLETE__FORMAL_ADJUDICATION_PENDING"),
    ("07", "07_h3_adjudication.ipynb", "Formal H3 adjudication", "COMPLETE / FORMALLY ADJUDICATED", "H3_SUPPLY_REDESIGN_INSUFFICIENT"),
]
index = pd.DataFrame(entries, columns=["step", "notebook", "purpose", "status", "scientific state"])
index["notebook SHA-256"] = [
    hashlib.sha256((ROOT / "notebooks/phasewise/post_phase14_h3_proposal_supply" / name).read_bytes()).hexdigest()
    if (ROOT / "notebooks/phasewise/post_phase14_h3_proposal_supply" / name).exists() else "PENDING"
    for _, name, *_ in entries
]
display(index)
print("Phase 14 is separate and complete; Phase 15 remains NOT_STARTED_AND_NOT_AUTHORIZED; TEST is CLOSED.")
print()
print("=== STEP 04 AUTHORITATIVE TRAINING CLOSURE ===")
closure = load_json("reports/audits/h3_proposal_supply_training_closure.json")
replay = load_json("reports/audits/h3_checkpoint_selection_replay.json")
ckpt = closure["checkpoint_integrity"]["best_pt"]
summary_rows = [
    ("training closure", "reports/audits/h3_proposal_supply_training_closure.json", closure["status"]),
    ("checkpoint-selection replay", "reports/audits/h3_checkpoint_selection_replay.json", replay["verdict"]),
    ("total completed epochs", closure["total_epochs"], closure["stop_reason"]),
    ("best epoch (frozen tuple)", closure["best_epoch"], closure["selection_rule"]),
    ("best val_localization_recall", closure["best_val_localization_recall"], ""),
    ("best val_loss", closure["best_val_loss"], ""),
    ("best checkpoint", ckpt["path"], ckpt["sha256"]),
]
display(pd.DataFrame(summary_rows, columns=["item", "value", "detail"]))
print("Notebook-04 executed SHA-256:", index.loc[index["step"] == "04", "notebook SHA-256"].iloc[0])

print()
print("=== STEP 05 SUPPLY-AXIS EVALUATION (RECONCILED) ===")
eval_art = load_json("reports/audits/h3_supply_axis_evaluation.json")
criteria = load_json("reports/audits/h3_proposal_supply_success_failure_criteria.json")
recon = load_json("reports/audits/h3_supply_axis_metric_domain_reconciliation.json")

h3m = eval_art["h3_metrics"]
r_h3_proxy = eval_art["r_h3"]
prim = eval_art["primary_criteria"]
fb = eval_art["failure_boundary"]

eval_rows = [
    ("evaluation artifact", "reports/audits/h3_supply_axis_evaluation.json", eval_art["status"]),
    ("metric-domain reconciliation", "reports/audits/h3_supply_axis_metric_domain_reconciliation.json", recon["status"]),
    ("selected epoch", eval_art["selected_epoch"], eval_art["checkpoint_selection_replay_verdict"]),
    ("VAL images / components", f'{eval_art["val_population"]["images"]} / {eval_art["val_population"]["components"]}', f'TINY={eval_art["val_population"]["tiny"]} SMALL={eval_art["val_population"]["small"]} MEDIUM={eval_art["val_population"]["medium"]} LARGE={eval_art["val_population"]["large"]}'),
    ("frozen decode", f'thr={eval_art["frozen_decode"]["threshold"]} margin={eval_art["frozen_decode"]["margin"]} top_k={eval_art["frozen_decode"]["top_k"]} nms={eval_art["frozen_decode"]["nms_iou"]}', "exactly 1 policy"),
    ("total proposals", h3m["proposal_counts"]["total_proposals"], f'{h3m["proposal_counts"]["proposals_per_image"]:.4f}/image'),
    ("zero-proposal GT", f'{h3m["proposal_counts"]["zero_proposal_gt"]}/152', f'baseline 79/152 -> {79-h3m["proposal_counts"]["zero_proposal_gt"]} recovered'),
    ("zero-proposal TINY", f'{h3m["zero_proposal_by_size"]["TINY"]}/72', f'baseline 57/72'),
    ("R_H3 (LOCALIZER-ONLY PROXY)", f'{r_h3_proxy["R_H3"]:.10f}', "NOT FORMAL (proposal domain) - see reconciliation"),
    ("Criterion A (TINY<=20/72)", f'{prim["criterion_a"]["value"]}/72', "PASS" if prim["criterion_a"]["passed"] else "FAIL"),
    ("Criterion B (GT<=35/152)", f'{prim["criterion_b"]["value"]}/152', "PASS" if prim["criterion_b"]["passed"] else "FAIL"),
    ("primary A AND B", "", "PASS" if prim["combined_primary_passed"] else "FAIL"),
    ("TEST reads", str(eval_art["test_reads"]), "CLOSED"),
]
display(pd.DataFrame(eval_rows, columns=["metric", "value", "detail"]))
print("Notebook-05 SHA-256:", hashlib.sha256((ROOT / "notebooks/phasewise/post_phase14_h3_proposal_supply" / "05_h3_supply_axis_evaluation.ipynb").read_bytes()).hexdigest())

print()
print("=== STEP 06 H3 FROZEN DOWNSTREAM EVALUATION ===")
down = load_json("reports/audits/h3_downstream_evaluation.json")
corr = load_json("reports/audits/h3_downstream_metric_domain_correction.json")
fd = down["formal_downstream"]
m = down["h3_downstream_metrics"]
ct = down["contrast_table"]
down_rows = [
    ("evaluation artifact", "reports/audits/h3_downstream_evaluation.json", down["status"]),
    ("metric-domain correction", "reports/audits/h3_downstream_metric_domain_correction.json", corr["status"]),
    ("selected epoch", down["selected_h3_checkpoint"]["selected_epoch"], f'heatmap_resolution={down["selected_h3_checkpoint"]["heatmap_resolution"]}'),
    ("frozen proposal policy", f'{down["frozen_proposal_policy"]}', "exactly 1 policy"),
    ("population", f'{down["population"]["val_images"]} images / {down["population"]["components"]} components', str(down["population"]["size_strata"])),
    ("TINY recall@90 (detection)", f'{fd["TinyRecall_H3_fraction"]} = {fd["TinyRecall_H3"]:.10f}', "formal frozen domain"),
    ("TinyRecall_H3 (cov90 sensitivity)", f'{m["descriptive_strict_cov90"]["tiny_cov90"]:.10f}', "descriptive only"),
    ("Tg / G", f'{fd["Tg"]} / {fd["G"]}', "frozen references"),
    ("R_H3 (FORMAL)", f'{fd["R_H3"]:.10f}', f'band: {fd["interpretation_band"]}'),
    ("R_H3 (cov90 sensitivity)", f'{fd["sensitivity_strict_cov90"]["R_H3_cov90"]:.10f}', "NOT formal"),
    ("overall recall@90 (detection)", f'{m["overall_recall90_det"]:.10f}', "frozen domain"),
    ("mean Dice", f'{m["mean_dice"]:.6f}', f'GLOBAL {ct["GLOBAL"]["mean_dice"]:.6f} / PREDICTED {ct["PREDICTED"]["mean_dice"]:.6f}'),
    ("hard-Tiny recovered", f'{m["hard_tiny_recovered"]}/15', f'cov90: {m["descriptive_strict_cov90"]["hard_tiny_recovered_cov90"]}/15'),
    ("global-empty recovered", f'{m["global_empty_recovered"]}/5', f'cov90: {m["descriptive_strict_cov90"]["global_empty_recovered_cov90"]}/5'),
    ("formal adjudication", str(down["formal_adjudication_performed"]), "NOT PERFORMED (Notebook 07 deferred)"),
    ("TEST reads", str(down["test_reads"]), "CLOSED"),
]
display(pd.DataFrame(down_rows, columns=["metric", "value", "detail"]))
print("Notebook-06 SHA-256:", hashlib.sha256((ROOT / "notebooks/phasewise/post_phase14_h3_proposal_supply" / "06_h3_downstream_evaluation.ipynb").read_bytes()).hexdigest())
print("Formal H3 adjudication remains PENDING (Notebook 07, not authorized). Phase 15 remains NOT_STARTED_AND_NOT_AUTHORIZED. TEST is CLOSED.")

print()
print("=== STEP 07 FORMAL H3 ADJUDICATION ===")
adj = load_json("reports/audits/h3_formal_adjudication.json")
cr = adj["criterion_results"]
fbx = adj["failure_boundary"]
adj_rows = [
    ("adjudication artifact", "reports/audits/h3_formal_adjudication.json", adj["status"]),
    ("scientific label", adj["scientific_label"], "FROZEN"),
    ("selected epoch", adj["selected_checkpoint"]["selected_epoch"], adj["checkpoint_selection_replay_verdict"]),
    ("selected checkpoint SHA", adj["selected_checkpoint"]["sha256"], f"matches expected: {adj['selected_checkpoint']['sha256_matches_expected']}"),
    ("Criterion A (TINY<=20/72)", cr["criterion_A"]["observed"], cr["criterion_A"]["result"]),
    ("Criterion B (GT<=35/152)", cr["criterion_B"]["observed"], cr["criterion_B"]["result"]),
    ("combined primary supply", "", cr["combined_primary_supply_result"]),
    ("formal TinyRecall_H3", adj["observed"]["formal_TinyRecall_H3_fraction"], f'{adj["observed"]["formal_TinyRecall_H3"]:.10f}'),
    ("formal R_H3", adj["observed"]["formal_R_H3_fraction"], f'{adj["observed"]["formal_R_H3"]:.10f}'),
    ("R_H3 band", adj["observed"]["R_H3_band"], f"insufficient threshold {adj['observed']['formal_TinyRecall_H3_fraction'] and 0.15}"),
    ("failure clause 1 (TINY>=45/72)", fbx["clause_1"]["observed"], "FIRED" if fbx["clause_1"]["fired"] else "not fired"),
    ("failure clause 2 (R_H3<0.15)", f'{fbx["clause_2"]["observed"]:.10f}', "FIRED" if fbx["clause_2"]["fired"] else "not fired"),
    ("failure boundary overall", "", "FIRED" if fbx["fired"] else "not fired"),
    ("Phase-15 status", adj["governance"]["phase_15_status"], "LOCKED"),
    ("TEST reads", str(adj["governance"]["test_reads"]), "CLOSED"),
]
display(pd.DataFrame(adj_rows, columns=["item", "value", "result"]))
print("Notebook-07 SHA-256:", hashlib.sha256((ROOT / "notebooks/phasewise/post_phase14_h3_proposal_supply" / "07_h3_adjudication.ipynb").read_bytes()).hexdigest())
print("FROZEN H3 LABEL:", adj["scientific_label"])
print("State: H3_FORMAL_ADJUDICATION_COMPLETE__TARGET_SUPPORT_INSUFFICIENT__PHASE15_LOCKED")


,step,notebook,purpose,status,scientific state,notebook SHA-256
0,01,01_proposal_root_cause_diagnostic.ipynb,proposal/localizer root-cause diagnostic,COMPLETE,F_MIXED_BOTTLENECK,a9fd0dd734f9311655677eadc3e8d1710d9725c01681c3...
1,02,02_h3_axis_priority_diagnostic.ipynb,A-only vs C-only causal priority,COMPLETE,H3_DESIGN_ELIGIBLE__AXIS_PROPOSAL_SUPPLY_RECALL,ee8787c6d76acd6fe0208a694104d8d4b9b65e6fc3ea90...
2,03,03_h3_design_and_pretraining_binding.ipynb,"H3 design, corrections, calibration, binding",COMPLETE,H3_FINAL_PRETRAINING_BINDING_COMPLETE__AWAITIN...,b98538a9d83e8da3dc696364f1cdb0a10bc07eb13b8c33...
3,04,04_h3_training.ipynb,H3 training record,COMPLETE,H3_TRAINING_COMPLETE__CHECKPOINT_SELECTION_REP...,83aabc82f682d6c5685812f815077bf83b068e84ce7311...
4,05,05_h3_supply_axis_evaluation.ipynb,H3 supply-axis evaluation (metric-domain RECON...,COMPLETE / RECONCILED,H3_SUPPLY_METRIC_DOMAIN_RECONCILED__DOWNSTREAM...,e3771f0a33cf8e69b0fe734001494d2974d1d896e540ce...
5,06,06_h3_downstream_evaluation.ipynb,H3 frozen downstream evaluation,COMPLETE,H3_DOWNSTREAM_EVALUATION_COMPLETE__FORMAL_ADJU...,557b8185edf2ab12e1332afdc3d346d8a4678346f3a9a1...
6,07,07_h3_adjudication.ipynb,Formal H3 adjudication,COMPLETE / FORMALLY ADJUDICATED,H3_SUPPLY_REDESIGN_INSUFFICIENT,08c3ec4216bfc14f5aa4eb2dc2f12b00caaf53203f7805...


Phase 14 is separate and complete; Phase 15 remains NOT_STARTED_AND_NOT_AUTHORIZED; TEST is CLOSED.

=== STEP 04 AUTHORITATIVE TRAINING CLOSURE ===


,item,value,detail
0,training closure,reports/audits/h3_proposal_supply_training_clo...,H3_TRAINING_COMPLETE__CHECKPOINT_SELECTION_REP...
1,checkpoint-selection replay,reports/audits/h3_checkpoint_selection_replay....,H3_CHECKPOINT_SELECTION_REPLAY_PASS
2,total completed epochs,69,EARLY_STOPPING_PATIENCE_15_FIRED
3,best epoch (frozen tuple),54,MAX val_localization_recall then MIN val_loss
4,best val_localization_recall,0.638158,
5,best val_loss,1.856133,
6,best checkpoint,D:\Softfors\jup\checkpoints\h3_kub_localizer_s...,d875cac7d09bbc3d32232191f0af2c138e5a74682fc63e...


Notebook-04 executed SHA-256: 83aabc82f682d6c5685812f815077bf83b068e84ce7311c51366d67883ed350d

=== STEP 05 SUPPLY-AXIS EVALUATION (RECONCILED) ===


,metric,value,detail
0,evaluation artifact,reports/audits/h3_supply_axis_evaluation.json,H3_SUPPLY_AXIS_EVALUATION_COMPLETE__FORMAL_ADJ...
1,metric-domain reconciliation,reports/audits/h3_supply_axis_metric_domain_re...,H3_SUPPLY_METRIC_DOMAIN_RECONCILED__DOWNSTREAM...
2,selected epoch,54,H3_CHECKPOINT_SELECTION_REPLAY_PASS
3,VAL images / components,103 / 152,TINY=72 SMALL=28 MEDIUM=36 LARGE=16
4,frozen decode,thr=0.1 margin=0.2 top_k=5 nms=0.3,exactly 1 policy
5,total proposals,512,4.9709/image
6,zero-proposal GT,27/152,baseline 79/152 -> 52 recovered
7,zero-proposal TINY,22/72,baseline 57/72
8,R_H3 (LOCALIZER-ONLY PROXY),-0.6000000000,NOT FORMAL (proposal domain) - see reconciliation
9,Criterion A (TINY<=20/72),22/72,FAIL


Notebook-05 SHA-256: e3771f0a33cf8e69b0fe734001494d2974d1d896e540ced958b730bca39a3198

=== STEP 06 H3 FROZEN DOWNSTREAM EVALUATION ===


,metric,value,detail
0,evaluation artifact,reports/audits/h3_downstream_evaluation.json,H3_DOWNSTREAM_EVALUATION_COMPLETE__FORMAL_ADJU...
1,metric-domain correction,reports/audits/h3_downstream_metric_domain_cor...,H3_DOWNSTREAM_METRIC_DOMAIN_CORRECTED__FORMAL_...
2,selected epoch,54,heatmap_resolution=128
3,frozen proposal policy,"{'threshold': 0.1, 'margin': 0.2, 'top_k': 5, ...",exactly 1 policy
4,population,103 images / 152 components,"{'TINY': 72, 'SMALL': 28, 'MEDIUM': 36, 'LARGE..."
5,TINY recall@90 (detection),59/72 = 0.8194444444,formal frozen domain
6,TinyRecall_H3 (cov90 sensitivity),0.5000000000,descriptive only
7,Tg / G,0.7916666666666666 / 0.20833333333333337,frozen references
8,R_H3 (FORMAL),0.1333333333,band: insufficient
9,R_H3 (cov90 sensitivity),-1.4000000000,NOT formal


Notebook-06 SHA-256: 557b8185edf2ab12e1332afdc3d346d8a4678346f3a9a16c501754ce839fa3ab
Formal H3 adjudication remains PENDING (Notebook 07, not authorized). Phase 15 remains NOT_STARTED_AND_NOT_AUTHORIZED. TEST is CLOSED.

=== STEP 07 FORMAL H3 ADJUDICATION ===


,item,value,result
0,adjudication artifact,reports/audits/h3_formal_adjudication.json,H3_FORMAL_ADJUDICATION_COMPLETE__TARGET_SUPPOR...
1,scientific label,H3_SUPPLY_REDESIGN_INSUFFICIENT,FROZEN
2,selected epoch,54,H3_CHECKPOINT_SELECTION_REPLAY_PASS
3,selected checkpoint SHA,d875cac7d09bbc3d32232191f0af2c138e5a74682fc63e...,matches expected: True
4,Criterion A (TINY<=20/72),22/72,FAIL
5,Criterion B (GT<=35/152),27/152,PASS
6,combined primary supply,,FAIL
7,formal TinyRecall_H3,59/72,0.8194444444
8,formal R_H3,2/15,0.1333333333
9,R_H3 band,insufficient,insufficient threshold 0.15


Notebook-07 SHA-256: 08c3ec4216bfc14f5aa4eb2dc2f12b00caaf53203f78056ffd1b57d7d7bcc66b
FROZEN H3 LABEL: H3_SUPPLY_REDESIGN_INSUFFICIENT
State: H3_FORMAL_ADJUDICATION_COMPLETE__TARGET_SUPPORT_INSUFFICIENT__PHASE15_LOCKED


## Authoritative chain

01 reads `phase14_diag_root_cause.json` and saved diagnostic tables.  
02 reads `post_phase14_h3_axis_priority_results.json` and saved A/C tables.  
03 reads the H3 design/correction/final-binding artifacts and calibration tables.  
04 is COMPLETE and executed: 69 epochs (early stopping, patience 15), best epoch 54 by the frozen (val_localization_recall, -val_loss) tuple, replay verdict H3_CHECKPOINT_SELECTION_REPLAY_PASS; best checkpoint `checkpoints/h3_kub_localizer_sigma_adaptive_sizeaware_seed42_clean_v3/best.pt` (SHA d875cac7d09bbc3d32232191f0af2c138e5a74682fc63e4d6e01f30345eca75e).  
05 is COMPLETE and RECONCILED: the supply-axis evaluation measured localizer/proposal supply (zero-proposal GT 79->27, zero-proposal TINY 57->22). A post-evaluation metric-domain reconciliation (`reports/audits/h3_supply_axis_metric_domain_reconciliation.json`) relabelled the earlier proposal-domain substitution as a LOCALIZER-ONLY PROXY and marked it INVALID_FOR_FORMAL_ADJUDICATION__PROPOSAL_DOMAIN_ONLY. State `H3_SUPPLY_METRIC_DOMAIN_RECONCILED__DOWNSTREAM_R_H3_PENDING`.  
06 is COMPLETE and executed: the single frozen H3 downstream evaluation (H3 localizer epoch 54 -> frozen Stage-4D proposal policy -> frozen Phase-14 refiner epoch 56 -> fusion -> segmentation). In the frozen detection domain (`>=1px overlap`, matching the frozen Tg), TinyRecall_H3 = 59/72 = 0.8194444444444444 and the formal R_H3 = (TinyRecall_H3 - Tg)/G = 2/15 = 0.13333333333333333 (insufficient). A metric-layer correction (`reports/audits/h3_downstream_metric_domain_correction.json`) aligned the aggregate to the frozen detection domain; the as-run cov90 artifact is preserved (`reports/audits/h3_downstream_evaluation_v1_asrun.json`). State `H3_DOWNSTREAM_EVALUATION_COMPLETE__FORMAL_ADJUDICATION_PENDING`.  
07 is COMPLETE and executed: the formal H3 adjudication applied the prospectively frozen criteria to the authoritative observed values. Criterion A (zero-proposal TINY <= 20/72) FAILED at 22/72; Criterion B (zero-proposal GT <= 35/152) PASSED at 27/152; the formal downstream R_H3 = 2/15 = 0.13333333333333333 < 0.15, so the frozen failure boundary FIRED. The frozen scientific label is **H3_SUPPLY_REDESIGN_INSUFFICIENT**. The intervention materially improved proposal supply (zero-proposal GT 79->27, TINY 57->22) but the recovered supply did not translate into sufficient downstream Tiny-recall gain under the frozen Stage-4D pipeline. Phase 15 remains NOT_STARTED_AND_NOT_AUTHORIZED; TEST remains CLOSED.

H3 is post-Phase14 work. It is not Phase 15 and does not authorize Phase 15.